In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from math import log, sqrt
from mhnlib.fixed_points import get_symmetric_stability_matrix_gram, get_entropies, get_jacobian_gram, get_symmetric_stability_matrices_gram
from mhnlib.dynamics import DualDeterministicDynamics, StochasticDynamics

In [2]:
%load_ext autoreload

In [18]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.cuda.empty_cache()
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

In [9]:
K = 256
N = 32
rho0 = 0.1
rho1 = 0.9
B = K // 8
M = K // B  # number of blocks
N_samples = 1
z_0 = torch.randn(N_samples, N) / sqrt(N)
z_block = torch.randn(N_samples, M, N) / sqrt(N)
eps = torch.randn(N_samples, K, N) / sqrt(N)

z_block_expanded = torch.repeat_interleave(z_block, repeats=B, dim=1)

all_patterns = (
    sqrt(rho0) * z_0[:, None, :]
    + sqrt(rho1 - rho0) * z_block_expanded
    + sqrt(1.0 - rho1) * eps)
all_centered_patterns = all_patterns - all_patterns.mean(dim=1, keepdims=True)
all_centered_grams = torch.einsum('ski,sli->skl', all_centered_patterns, all_centered_patterns)

In [45]:
sample_idx = 0
gram = all_centered_grams[sample_idx]
w0 = torch.ones(K)/K
dd_dyn = DualDeterministicDynamics(all_centered_patterns[sample_idx]).to(device)
stab_0, proj_0, infos_0 = get_symmetric_stability_matrices_gram(gram, w0, return_fisher=True, return_proj=True)
C_invsqrt = infos_0['fisher_invsqrt'][0]
stab_0 = stab_0[0]
proj_0 = proj_0[0]
stab_0_vals, stab_0_vecs = torch.linalg.eigh(stab_0)
beta_c = 1.0/stab_0_vals[-1]
beta_c

tensor(6.1694)

In [ ]:
max_num_eigenvectors = 10
num_trials=1000
perturbation_std = 1.0
delta_beta =0.001
for num_ev in range(1, max_num_eigenvectors+1):
    U = stab_0_vecs[:, -num_ev:]
    coeffs = torch.randn((num_trials, num_ev))*perturbation_std
    logits_perturbations = (coeffs @ U.T ) @ C_invsqrt.T
    logits_perturbations -= logits_perturbations.mean(dim=1, keepdim=True)
    logits_0 = torch.log(w0)
    logits_ic = logits_0[None, :] + logits_perturbations
    w_ics = torch.softmax(h, dim=-1)
    candidate_fps = trial_w_fps = dd_dyn.integrate(w_ics.to(device), (beta_c + delta_beta).view(1).to(device) , num_iterations=5000, verbose=True).cpu()[:,0,:]
    candidate_fps_stab = get_symmetric_stability_matrices_gram(gram, candidate_fps)
    candidate_fps_stab_vals = torch.linalg.eigvalsh(candidate_fps_stab)

    candidates_overlap = candidate_fps @ candidate_fps.T
    candidates_sim = candidates_overlap > 1/K
    candidates_sim[torch.arange(0,candidates_sim.shape[0]), torch.arange(0,candidates_sim.shape[0])] = True
    break